# 06 Read Results

Goal:

1. scan the current `output/part1/` tree directly
2. collect final metrics from all `results.json`
3. collect convergence metrics from experiment logs
4. export clean csv summaries into `results/part1/`

This notebook is designed for the current 18-experiment output structure and does not depend on a manifest.


In [13]:
from pathlib import Path
import json
import re
import pandas as pd

CV_ROOT = Path('~/CV_Project').expanduser()
OUTPUT_ROOT = CV_ROOT / 'output' / 'part1'
RESULTS_ROOT = CV_ROOT / 'results' / 'part1'
FINAL_DIR = RESULTS_ROOT / 'final'
CONV_DIR = RESULTS_ROOT / 'convergence'

for d in [FINAL_DIR, CONV_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('OUTPUT_ROOT =', OUTPUT_ROOT)
print('RESULTS_ROOT =', RESULTS_ROOT)


OUTPUT_ROOT = /home/bzhang512/CV_Project/output/part1
RESULTS_ROOT = /home/bzhang512/CV_Project/results/part1


## 1. Scan all results.json files


In [14]:
result_files = sorted(OUTPUT_ROOT.rglob('results.json'))
print('num results.json =', len(result_files))
for p in result_files[:10]:
    print(p)
if len(result_files) > 10:
    print('...')


num results.json = 18
/home/bzhang512/CV_Project/output/part1/405841/planA/colmap_108/3dgs/default_40k_eval2k/results.json
/home/bzhang512/CV_Project/output/part1/405841/planA/colmap_108/scaffold/final_40k_eval2k/results.json
/home/bzhang512/CV_Project/output/part1/405841/planA/colmap_full/3dgs/default_40k_eval2k/results.json
/home/bzhang512/CV_Project/output/part1/405841/planA/colmap_full/scaffold/final_40k_eval2k/results.json
/home/bzhang512/CV_Project/output/part1/405841/planB/vggt_108/3dgs/default_40k_eval2k/results.json
/home/bzhang512/CV_Project/output/part1/405841/planB/vggt_108/scaffold/final_40k_eval2k/results.json
/home/bzhang512/CV_Project/output/part1/DL3DV-2/planA/colmap_108/3dgs/default_40k_eval2k/results.json
/home/bzhang512/CV_Project/output/part1/DL3DV-2/planA/colmap_108/scaffold/final_40k_eval2k/results.json
/home/bzhang512/CV_Project/output/part1/DL3DV-2/planA/colmap_full/3dgs/default_40k_eval2k/results.json
/home/bzhang512/CV_Project/output/part1/DL3DV-2/planA/colma

## 2. Parse final metrics from results.json


In [15]:
def flatten_dict(d, prefix=''):
    out = {}
    for k, v in d.items():
        key = f'{prefix}.{k}' if prefix else k
        if isinstance(v, dict):
            out.update(flatten_dict(v, key))
        else:
            out[key] = v
    return out

def parse_result_identity(path: Path):
    rel = path.relative_to(OUTPUT_ROOT)
    parts = rel.parts
    # expected: scene / plan / variant / model / run_name / results.json
    if len(parts) < 6:
        return None
    scene, plan, variant, model, run_name = parts[:5]
    experiment_name = '__'.join([scene, plan, variant, model, run_name])
    return {
        'scene': scene,
        'plan': plan,
        'variant': variant,
        'model': model,
        'run_name': run_name,
        'experiment_name': experiment_name,
        'result_path': str(path),
    }

final_rows = []
for path in result_files:
    identity = parse_result_identity(path)
    if identity is None:
        continue
    payload = json.loads(path.read_text(encoding='utf-8'))
    row = dict(identity)
    row.update(flatten_dict(payload))
    final_rows.append(row)

final_df = pd.DataFrame(final_rows)
print('final_df shape =', final_df.shape)
final_df.head()


final_df shape = (18, 10)


,scene,plan,variant,model,run_name,experiment_name,result_path,ours_40000.SSIM,ours_40000.PSNR,ours_40000.LPIPS
0,405841,planA,colmap_108,3dgs,default_40k_eval2k,405841__planA__colmap_108__3dgs__default_40k_e...,/home/bzhang512/CV_Project/output/part1/405841...,0.921299,31.350622,0.140309
1,405841,planA,colmap_108,scaffold,final_40k_eval2k,405841__planA__colmap_108__scaffold__final_40k...,/home/bzhang512/CV_Project/output/part1/405841...,0.918527,30.863672,0.148838
2,405841,planA,colmap_full,3dgs,default_40k_eval2k,405841__planA__colmap_full__3dgs__default_40k_...,/home/bzhang512/CV_Project/output/part1/405841...,0.937703,32.770302,0.125829
3,405841,planA,colmap_full,scaffold,final_40k_eval2k,405841__planA__colmap_full__scaffold__final_40...,/home/bzhang512/CV_Project/output/part1/405841...,0.933910,32.061390,0.135362
4,405841,planB,vggt_108,3dgs,default_40k_eval2k,405841__planB__vggt_108__3dgs__default_40k_eval2k,/home/bzhang512/CV_Project/output/part1/405841...,0.849524,27.503836,0.201963


## 3. Locate matching log files


In [16]:
def find_log_file(row):
    log_dir = OUTPUT_ROOT / row['scene'] / row['plan'] / row['variant'] / row['model'] / 'logs'
    if not log_dir.exists():
        return None
    candidates = sorted(log_dir.glob('*.log'))
    if not candidates:
        return None

    tokens = [row['scene'], row['plan'], row['variant'], row['model']]
    for c in candidates:
        name = c.name
        if all(tok in name for tok in tokens):
            return c

    if len(candidates) == 1:
        return candidates[0]
    return None

if not final_df.empty:
    final_df['log_path'] = final_df.apply(find_log_file, axis=1)
    final_df['log_path'] = final_df['log_path'].astype('string')

final_df[['experiment_name', 'log_path']].head()


,experiment_name,log_path
0,405841__planA__colmap_108__3dgs__default_40k_e...,/home/bzhang512/CV_Project/output/part1/405841...
1,405841__planA__colmap_108__scaffold__final_40k...,/home/bzhang512/CV_Project/output/part1/405841...
2,405841__planA__colmap_full__3dgs__default_40k_...,/home/bzhang512/CV_Project/output/part1/405841...
3,405841__planA__colmap_full__scaffold__final_40...,/home/bzhang512/CV_Project/output/part1/405841...
4,405841__planB__vggt_108__3dgs__default_40k_eval2k,/home/bzhang512/CV_Project/output/part1/405841...


## 4. Read convergence metrics from logs


In [17]:
pattern = re.compile(r'\[ITER\s+(\d+)\]\s+Evaluating\s+(test|train):\s+L1\s+([0-9eE.+-]+)\s+PSNR\s+([0-9eE.+-]+)')

conv_rows = []
for _, row in final_df.iterrows():
    log_path = row.get('log_path')
    if pd.isna(log_path) or not log_path:
        continue
    log_path = Path(log_path)
    if not log_path.exists():
        continue
    with log_path.open('r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            m = pattern.search(line)
            if not m:
                continue
            conv_rows.append({
                'experiment_name': row['experiment_name'],
                'scene': row['scene'],
                'plan': row['plan'],
                'variant': row['variant'],
                'model': row['model'],
                'run_name': row['run_name'],
                'iter': int(m.group(1)),
                'split': m.group(2),
                'l1': float(m.group(3)),
                'psnr': float(m.group(4)),
                'log_path': str(log_path),
            })

conv_df = pd.DataFrame(conv_rows)
print('conv_df shape =', conv_df.shape)
conv_df.head()


conv_df shape = (720, 11)


,experiment_name,scene,plan,variant,model,run_name,iter,split,l1,psnr,log_path
0,405841__planA__colmap_108__3dgs__default_40k_e...,405841,planA,colmap_108,3dgs,default_40k_eval2k,2000,test,0.026272,26.110017,/home/bzhang512/CV_Project/output/part1/405841...
1,405841__planA__colmap_108__3dgs__default_40k_e...,405841,planA,colmap_108,3dgs,default_40k_eval2k,2000,train,0.024606,26.676241,/home/bzhang512/CV_Project/output/part1/405841...
2,405841__planA__colmap_108__3dgs__default_40k_e...,405841,planA,colmap_108,3dgs,default_40k_eval2k,4000,test,0.022073,27.561419,/home/bzhang512/CV_Project/output/part1/405841...
3,405841__planA__colmap_108__3dgs__default_40k_e...,405841,planA,colmap_108,3dgs,default_40k_eval2k,4000,train,0.020803,28.068633,/home/bzhang512/CV_Project/output/part1/405841...
4,405841__planA__colmap_108__3dgs__default_40k_e...,405841,planA,colmap_108,3dgs,default_40k_eval2k,6000,test,0.019645,28.582794,/home/bzhang512/CV_Project/output/part1/405841...


## 5. Export csv summaries


In [18]:
final_csv = FINAL_DIR / 'final_metrics_18.csv'
conv_all_csv = CONV_DIR / 'all_convergence_metrics.csv'

if not final_df.empty:
    final_df.to_csv(final_csv, index=False)
if not conv_df.empty:
    conv_df.to_csv(conv_all_csv, index=False)

if not conv_df.empty:
    for exp_name, subdf in conv_df.groupby('experiment_name'):
        out = CONV_DIR / f'{exp_name}.csv'
        subdf.sort_values(['split', 'iter']).to_csv(out, index=False)

print('final csv =', final_csv)
print('convergence all csv =', conv_all_csv)
print('per-experiment convergence dir =', CONV_DIR)


final csv = /home/bzhang512/CV_Project/results/part1/final/final_metrics_18.csv
convergence all csv = /home/bzhang512/CV_Project/results/part1/convergence/all_convergence_metrics.csv
per-experiment convergence dir = /home/bzhang512/CV_Project/results/part1/convergence


## 6. Quick checks


In [ ]:
print('num final experiments =', len(final_df))
if not final_df.empty:
    display(final_df[['scene', 'plan', 'variant', 'model', 'run_name']].sort_values(['scene', 'plan', 'variant', 'model']))

if not conv_df.empty:
    display(conv_df.groupby(['experiment_name', 'split'])['iter'].agg(['min', 'max', 'count']).reset_index())
else:
    print('No convergence rows parsed.')


num final experiments = 18


,scene,plan,variant,model,run_name
0,405841,planA,colmap_108,3dgs,default_40k_eval2k
1,405841,planA,colmap_108,scaffold,final_40k_eval2k
2,405841,planA,colmap_full,3dgs,default_40k_eval2k
3,405841,planA,colmap_full,scaffold,final_40k_eval2k
4,405841,planB,vggt_108,3dgs,default_40k_eval2k
5,405841,planB,vggt_108,scaffold,final_40k_eval2k
6,DL3DV-2,planA,colmap_108,3dgs,default_40k_eval2k
7,DL3DV-2,planA,colmap_108,scaffold,final_40k_eval2k
8,DL3DV-2,planA,colmap_full,3dgs,default_40k_eval2k
9,DL3DV-2,planA,colmap_full,scaffold,final_40k_eval2k


,experiment_name,split,min,max,count
0,405841__planA__colmap_108__3dgs__default_40k_e...,test,2000,40000,20
1,405841__planA__colmap_108__3dgs__default_40k_e...,train,2000,40000,20
2,405841__planA__colmap_108__scaffold__final_40k...,test,2000,40000,20
3,405841__planA__colmap_108__scaffold__final_40k...,train,2000,40000,20
4,405841__planA__colmap_full__3dgs__default_40k_...,test,2000,40000,20
5,405841__planA__colmap_full__3dgs__default_40k_...,train,2000,40000,20
6,405841__planA__colmap_full__scaffold__final_40...,test,2000,40000,20
7,405841__planA__colmap_full__scaffold__final_40...,train,2000,40000,20
8,405841__planB__vggt_108__3dgs__default_40k_eval2k,test,2000,40000,20
9,405841__planB__vggt_108__3dgs__default_40k_eval2k,train,2000,40000,20


: 